# Framework modular de experimentos — madurez de paltas

Este notebook es un **orquestador**, no un contenedor de lógica. Las operaciones reutilizables viven en `src/avocado/`; aquí configuramos una corrida, inspeccionamos sus datos y conectamos el modelo que queramos probar.

Principios:

- una palta (`sample_id`) pertenece a un único split;
- el nombre del archivo y el día no se usan como features;
- toda corrida tiene configuración, semilla, métricas y artefactos trazables;
- cambiar un experimento debe requerir editar principalmente la celda **Configuración**.

## 1. Inicialización

Ejecuta primero `python scripts/run_eda.py` si todavía no existe el manifiesto.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

# Durante el desarrollo, evita importar una versión antigua de src/avocado
# que haya quedado cacheada tras modificar los módulos.
for module_name in list(sys.modules):
    if module_name == 'avocado' or module_name.startswith('avocado.'):
        del sys.modules[module_name]

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from avocado import (
    AugmentationConfig, CrossValidationConfig, DataConfig, ExperimentConfig, SplitConfig,
    assert_no_group_leakage, assert_view_pairs, assign_cross_validation_folds,
    cross_validation_summary, grouped_split, load_experiment_data,
    log_result, save_experiment_config, split_summary,
    train_experiment, cross_validate_experiment, evaluate_checkpoint,
    cross_validate_handcrafted_baseline, hardware_preflight,
    load_loss_histories, plot_augmentation_preview, plot_loss_evolution,
    select_best_validation, MODEL_REGISTRY, MODEL_PROFILE_16GB,
    cuda_diagnostics, resolve_device,
)
from dataclasses import replace

pd.set_option('display.max_columns', 50)

### Verificación obligatoria de CUDA

La GPU es NVIDIA Blackwell. PyTorch debe tener un wheel CUDA 13 instalado **en este mismo kernel**. Si es necesario, ejecuta `%pip install torch==2.12.1 torchvision==0.27.1 --index-url https://download.pytorch.org/whl/cu130`, reinicia el kernel y vuelve a ejecutar desde el comienzo.

In [ ]:
CUDA_INFO = cuda_diagnostics()
display(pd.Series(CUDA_INFO, name='valor').to_frame())
DEVICE = resolve_device('cuda')  # aborta; nunca hace fallback silencioso a CPU
print(f'CUDA verificada: {DEVICE}')

## 2. Configuración del experimento

Esta es la celda principal que se modifica entre corridas. `model_name` y sus hiperparámetros quedan registrados aunque el backend de entrenamiento se conectará en la sección 6.

In [ ]:
CONFIG = ExperimentConfig(
    name='baseline_v001',
    data=DataConfig(
        manifest_path=ROOT / 'reports' / 'eda' / 'manifest.csv',
        valid_only=True,
        classes=(1, 2, 3, 4, 5),
    ),
    split=SplitConfig(
        train_fraction=0.70,
        validation_fraction=0.10,
        test_fraction=0.20,
        seed=42,
    ),
    model_name='resnet18',  # resnet18, efficientnet_b0 o dino_vits16
    image_size=224,
    batch_size=32,
    epochs=10,
    learning_rate=3e-4,
    weight_decay=1e-4,
    patience=3,
    accelerator='cuda',
    mixed_precision=True,
    notes='Comparación inicial con transfer learning.',
)

CONFIG.to_dict()

In [ ]:
CV_CONFIG = CrossValidationConfig(
    n_splits=5,
    seed=42,
    selection_metric='macro_f1',
)

## 3. Dataset y split agrupado

El split se realiza dentro de cada condición de almacenamiento (`T10`, `T20`, `Tam`) y siempre por `sample_id`. Todas las vistas y todos los días de una palta permanecen juntos.

In [ ]:
dataset = load_experiment_data(CONFIG.data)
dataset = grouped_split(dataset, CONFIG.data, CONFIG.split)
assert_no_group_leakage(dataset, CONFIG.data.group_column)
assert_view_pairs(dataset)

split_sizes, class_distribution = split_summary(
    dataset, CONFIG.data.target_column, CONFIG.data.group_column
)
display(split_sizes)
display(class_distribution.rename_axis(columns='clase (%)'))

### Data augmentation

Esta celda controla las transformaciones aplicadas **solo a training**. Validation y test usan resize y center crop deterministas. Los cambios de color se mantienen moderados porque el color de la piel contiene señal de madurez.

In [ ]:
AUGMENTATION = AugmentationConfig(
    enabled=True,
    crop_scale_min=0.80,
    horizontal_flip_probability=0.50,
    rotation_degrees=10.0,
    brightness=0.12,
    contrast=0.12,
    saturation=0.08,
)
CONFIG = replace(CONFIG, augmentation=AUGMENTATION)

preview_row = dataset[dataset['split'] == 'train'].sample(
    1, random_state=CONFIG.split.seed
).iloc[0]
augmentation_figure, _ = plot_augmentation_preview(
    preview_row.image_path, CONFIG.image_size, AUGMENTATION, examples=6,
    seed=CONFIG.split.seed,
)
display(augmentation_figure)

In [ ]:
# Comprobaciones que deben pasar antes de cualquier entrenamiento.
assert dataset['image_ok'].all()
assert dataset['image_path'].map(Path).map(Path.is_file).all()
assert dataset.groupby('sample_id')['split'].nunique().max() == 1
assert dataset.groupby('pair_id')['split'].nunique().max() == 1
assert set(dataset['ripening_index']) == set(CONFIG.data.classes)

print(f"{len(dataset):,} imágenes válidas")
print(f"{dataset['sample_id'].nunique():,} paltas independientes")
print('Sin fuga de paltas ni de pares a/b entre splits')

## 4. Diagnóstico del split

Estas visualizaciones permiten verificar que una nueva semilla o filtro no haya creado una partición problemática.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
pd.crosstab(dataset['split'], dataset['ripening_index']).plot.bar(ax=axes[0])
axes[0].set(title='Clases por split', xlabel='Split', ylabel='Imágenes')
axes[0].tick_params(axis='x', rotation=0)

pd.crosstab(dataset['split'], dataset['storage_group']).plot.bar(ax=axes[1])
axes[1].set(title='Almacenamiento por split', xlabel='Split', ylabel='Imágenes')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()

## 5. Inspección visual reproducible

Cambia `SPLIT_TO_INSPECT` o `EXAMPLES_PER_CLASS` sin tocar la lógica de carga.

In [ ]:
SPLIT_TO_INSPECT = 'train'
EXAMPLES_PER_CLASS = 3

sample = (dataset[dataset['split'] == SPLIT_TO_INSPECT]
          .groupby('ripening_index', group_keys=False)
          .sample(n=EXAMPLES_PER_CLASS, random_state=CONFIG.split.seed))

fig, axes = plt.subplots(len(CONFIG.data.classes), EXAMPLES_PER_CLASS,
                         figsize=(3 * EXAMPLES_PER_CLASS, 3 * len(CONFIG.data.classes)))
for axis, row in zip(axes.ravel(), sample.itertuples(index=False)):
    with Image.open(row.image_path) as image:
        axis.imshow(image)
    axis.set_title(f'clase={row.ripening_index} | {row.storage_group} | d{row.day}')
    axis.axis('off')
plt.tight_layout()

## 6. Holdout simple opcional (deshabilitado)

Este flujo de validación única se conserva solo para pruebas rápidas. La comparación principal se realiza con cross-validation en la sección 7. Todos los modelos usan la misma cabeza de cinco clases y las mismas transformaciones.

Las features permitidas son únicamente los píxeles. `file_stem`, `filename_label`, `day`, `sample_id` y `storage_group` sirven para auditoría, no como entradas del modelo.

In [ ]:
MODEL_SUITE = (
    'resnet50',
    'efficientnetv2_s',
    'convnext_small',
    'swin_tiny',
    'dino_vits16',
    'dino_vitb16',
)
pd.DataFrame([
    {
        'alias': alias, 'timm_model': MODEL_REGISTRY[alias],
        **MODEL_PROFILE_16GB[alias],
        'effective_batch': (MODEL_PROFILE_16GB[alias]['batch_size']
                            * MODEL_PROFILE_16GB[alias]['gradient_accumulation_steps']),
    }
    for alias in MODEL_SUITE
])

In [ ]:
# Una minibatch por modelo: valida compatibilidad y VRAM antes de los 30 entrenamientos.
RUN_HARDWARE_PREFLIGHT = True
hardware_report = pd.DataFrame()
if RUN_HARDWARE_PREFLIGHT:
    hardware_report = hardware_preflight(
        MODEL_SUITE, MODEL_PROFILE_16GB, num_classes=len(CONFIG.data.classes)
    )
    display(hardware_report)
    if not hardware_report['status'].eq('ok').all():
        raise RuntimeError('El preflight detectó modelos incompatibles u OOM; revisa la tabla')

In [ ]:
# Cambia a True para lanzar las tres pruebas. El test NO se evalúa aquí.
RUN_TRAINING = False
validation_runs = []
validation_rows = []
validation_configs = {}

if RUN_TRAINING:
    for model_name in MODEL_SUITE:
        model_config = replace(
            CONFIG, name=f'{CONFIG.name}_{model_name}', model_name=model_name,
            **MODEL_PROFILE_16GB[model_name],
        )
        result = train_experiment(
            model_config, dataset, ROOT / 'experiments' / model_config.name,
            evaluate_test=False, num_workers=4,
        )
        validation_runs.append(result)
        validation_configs[model_name] = model_config
        validation_rows.append({
            'model': model_name, 'best_epoch': result.best_epoch,
            **result.validation_metrics,
        })

validation_table = pd.DataFrame(validation_rows).sort_values(
    'macro_f1', ascending=False
) if validation_rows else pd.DataFrame()
validation_table

### Evaluación final bloqueada

Selecciona el modelo usando solamente `validation_table`. Después ejecuta una única evaluación con `evaluate_test=True`; no uses el test para ajustar hiperparámetros.

In [ ]:
RUN_TESTING = False
test_metrics = None

if RUN_TESTING:
    best_run = select_best_validation(
        validation_runs, metric=CV_CONFIG.selection_metric
    )
    best_config = validation_configs[best_run.model_name]
    print('Modelo elegido exclusivamente por validación:', best_run.model_name)
    test_metrics = evaluate_checkpoint(
        best_config, dataset, best_run.checkpoint_path, split='test', num_workers=4
    )
    log_result(
        best_config, test_metrics, ROOT / 'experiments' / 'registry.jsonl',
        artifacts={'checkpoint': best_run.checkpoint_path, 'selection': 'fixed_validation'},
    )
test_metrics

## 7. Cross-validation agrupada

El test fijo del 20% recibe `cv_fold=-1` y nunca entra a los folds. El 80% de desarrollo se divide en cinco folds por `sample_id`, estratificados por almacenamiento. Cada arquitectura se entrena cinco veces y se compara mediante la media de validación.

In [ ]:
folded_dataset = assign_cross_validation_folds(
    dataset, CONFIG.data, n_splits=CV_CONFIG.n_splits, seed=CV_CONFIG.seed
)
display(cross_validation_summary(folded_dataset))
assert folded_dataset.loc[folded_dataset['split'] == 'test', 'cv_fold'].eq(-1).all()

### Baseline de color, textura y forma

Implementa la propuesta del experimento anterior: segmentación aproximada del fondo, descriptores RGB/HSV, forma y gradientes, seguidos por regresión logística. Usa exactamente los mismos folds y sirve para medir cuánto aportan las redes frente a señales visuales simples.

In [ ]:
RUN_HANDCRAFTED_BASELINE = True
handcrafted_results = pd.DataFrame()
if RUN_HANDCRAFTED_BASELINE:
    handcrafted_results = cross_validate_handcrafted_baseline(
        folded_dataset,
        cache_path=ROOT / 'experiments' / 'handcrafted_features.csv',
        seed=CV_CONFIG.seed,
    )
    display(handcrafted_results)
    display(handcrafted_results.groupby('model')[
        ['accuracy', 'balanced_accuracy', 'macro_f1', 'mae', 'qwk']
    ].agg(['mean', 'std']))

In [ ]:
# 6 arquitecturas x 5 folds = 30 entrenamientos.
RUN_CROSS_VALIDATION = True
cv_tables = []

if RUN_CROSS_VALIDATION:
    for model_name in MODEL_SUITE:
        cv_model_config = replace(
            CONFIG, name=f'cv_{CONFIG.name}_{model_name}', model_name=model_name,
            **MODEL_PROFILE_16GB[model_name],
        )
        _, fold_table = cross_validate_experiment(
            cv_model_config, folded_dataset,
            ROOT / 'experiments' / cv_model_config.name, num_workers=4,
        )
        cv_tables.append(fold_table)

cv_results = pd.concat(cv_tables, ignore_index=True) if cv_tables else pd.DataFrame()
if not cv_results.empty:
    metric_columns = ['accuracy', 'balanced_accuracy', 'macro_f1', 'mae', 'qwk']
    cv_aggregate = cv_results.groupby('model')[metric_columns].agg(['mean', 'std'])
else:
    cv_aggregate = pd.DataFrame()
cv_aggregate

In [ ]:
# Curvas agregadas de convergencia; la banda representa ±1 desviación entre folds.
if not cv_results.empty:
    loss_figure, _ = plot_loss_evolution(
        cv_results, ROOT / 'experiments' / 'cv_loss_evolution.png'
    )
    display(loss_figure)

In [ ]:
# Tras seleccionar por la media CV, se hace una sola corrida final con test.
RUN_CV_FINAL_TEST = False
cv_final_result = None

if RUN_CV_FINAL_TEST:
    best_cv_model = cv_aggregate[CV_CONFIG.selection_metric]['mean'].idxmax()
    final_config = replace(
        CONFIG, name=f'cv_selected_{best_cv_model}', model_name=best_cv_model,
        **MODEL_PROFILE_16GB[best_cv_model],
    )
    cv_final_result = train_experiment(
        final_config, dataset, ROOT / 'experiments' / final_config.name,
        evaluate_test=True, num_workers=4,
    )
    print('Modelo seleccionado por CV:', best_cv_model)
    log_result(
        final_config, cv_final_result.test_metrics,
        ROOT / 'experiments' / 'registry.jsonl',
        artifacts={'checkpoint': cv_final_result.checkpoint_path, 'selection': 'cross_validation'},
    )
cv_final_result.test_metrics if cv_final_result else None

## 8. Trazabilidad de la corrida

La configuración puede guardarse antes de entrenar. Cuando existan métricas reales, el último bloque las agregará a `experiments/registry.jsonl` sin sobrescribir corridas anteriores.

In [ ]:
RUN_DIR = ROOT / 'experiments' / CONFIG.name
config_path = save_experiment_config(CONFIG, RUN_DIR)
splits_path = RUN_DIR / 'splits.csv'
dataset[['file_stem', 'sample_id', 'pair_id', 'side', 'ripening_index', 'split']].to_csv(splits_path, index=False)

print('Configuración:', config_path)
print('Split congelado:', splits_path)

In [ ]:
# Ejecutar solamente después de obtener métricas reales.
# metrics = {'macro_f1': 0.0, 'balanced_accuracy': 0.0, 'mae': 0.0, 'qwk': 0.0}
# log_result(
#     CONFIG, metrics, ROOT / 'experiments' / 'registry.jsonl',
#     artifacts={'run_dir': str(RUN_DIR)},
# )

## Baseline propuesto: implementado

El baseline lineal sobre descriptores segmentados ya forma parte de la cross-validation. La siguiente extensión recomendable es comparar esta máscara aproximada con una segmentación aprendida y evaluar si eliminar completamente el fondo mejora la generalización. El conjunto de test permanece intacto hasta elegir la configuración final.